# Governance back-test: thresholds run against the CFPB complaint project

This notebook tests the thresholds in our governance memo against our own results. For each rule we ask two things: how often it fires, and whether it catches the errors we care about.

The data is the held-out set: 30 complaints with CFPB labels as the reference, plus one adversarial complaint. We ran three prompts (A, B and C). The notebook reads the cached outputs in `data/processed/holdout/`, so it makes no API calls. Run it from the `notebooks/` folder.

One caveat for everything below: we picked the thresholds after looking at these same 30 complaints. The results are in-sample, so they are probably optimistic.

## Summary: rules tested and results

Here are the rules in plain English.

**Rule 1: accuracy**
- Accuracy gate: a person checks every AI category until audit accuracy is at least 80% on 200 double-coded complaints, with a lower bound of at least 70%. We can't test this yet. It needs the audit.
- Low-F1 trigger: any complaint whose predicted category has an F1 below 0.50 goes to a person.

**Rule 2: human review**
- Kappa gate: no autonomous routing while blind kappa on the categories is below 0.60.
- The judge can't sign off alone: a brief goes to a person when the judge total and the human mean differ by at least the MDD (2.14 points).
- Narrative triggers: a complaint with fewer than 10 words, or with instruction-like text, goes to a person.

**Rule 3: escalation**
- Count reconciliation: a brief is blocked if a count or complaint ID in its table or bullet list doesn't match the classifications. Counts written in prose aren't checked. That's recommended for further analysis.
- Rubric total: a brief is blocked if its judge total is below 15 of 20.

**Data rules**
- Identifier scan: a pattern check for SSN-like and card-number-like strings. It's a double check, not a control for internal data.
- Duplicate IDs: a batch with duplicate complaint IDs is blocked.

## Back-test results

These are on the held-out set (30 complaints plus one adversarial), so they're in-sample and probably optimistic.

| Rule | Fires on | Errors caught | Decision |
|---|---|---|---|
| Low-F1 trigger, T = 0.50 | 12 of 30 predictions (40%) | 8 of 15 (53%) | Kept |
| Narrative under 10 words | 0 of 30 held-out; 0.40% of 18,276 | 0 of 15 | Kept as a guardrail; monthly random audit added |
| Injection screen | 3 of 31 (10%) | 1 of 1 adversarial; 2 unverified | Kept |
| Count reconciliation | Blocks 1 of 3 briefs (A) | 1 of 2 human-noted count errors; prose error missed | Prose counts outside this check; recommended for further analysis |
| Judge total below 15 | Blocks 2 of 3 briefs (A, B), on every run | Catches both briefs with human mean below 15 | Threshold moved from 17 to 15 after the run check |
| Judge-human gap of 2.14 or more | 1 of 3 briefs (C) | 1 of 1 judge miss beyond the MDD | Kept |
| Kappa gate (0.60) | Blocks all volume (kappa 0.272) | Not testable per complaint | Kept |
| Identifier scan | 0 of 31 public narratives | Not tested on planted identifiers | Double check only |
| Duplicate IDs | 0 of 18,276 | n/a | Kept |

In [1]:
import os
from pathlib import Path
import json, re
import pandas as pd

ROOT = Path(os.getcwd()).parent
H = ROOT / "data" / "processed" / "holdout"
RAW = ROOT / "data" / "raw" / "CFPB_credit_card_complaints_with_narratives_2015_2017.xlsx"

gen = json.loads((H / "generation_results.json").read_text(encoding="utf8"))
judge = json.loads((H / "judge_results.json").read_text(encoding="utf8"))
key = pd.read_csv(H / "holdout_answer_key.csv", dtype={"complaint_id": str})
cases = pd.read_csv(H / "holdout_cases.csv", dtype={"complaint_id": str})

typical = cases[cases.case_type == "typical"].merge(
    key[["complaint_id", "category"]], on="complaint_id", validate="one_to_one")
assert len(typical) == 30
typical = typical.rename(columns={"category": "true"})
typical["word_count"] = typical.narrative.str.split().str.len()

def eval_frame(prompt):
    """One row per held-out complaint: CFPB label, the prompt's prediction, error flag."""
    preds = pd.DataFrame(gen[prompt]["classifications"])
    preds["complaint_id"] = preds["id"].astype(str)
    ev = typical.merge(preds[["complaint_id", "category"]].rename(columns={"category": "predicted"}),
                       on="complaint_id", validate="one_to_one")
    ev["is_error"] = ev.true != ev.predicted
    return ev

print("Held-out complaints:", len(typical), "| Prompt C errors:", int(eval_frame("C").is_error.sum()))

Held-out complaints: 30 | Prompt C errors: 15


## Rule 1 back-test: low-F1 category trigger

A complaint goes to a person when its predicted category has a per-category F1 below T. The F1 comes from the Week 6 metrics, for Prompt C. We test T = 0.40, 0.50 and 0.60.

In [2]:
evC = eval_frame("C")
rows = []
for c in sorted(set(evC.true) | set(evC.predicted)):
    tp = ((evC.predicted == c) & (evC.true == c)).sum()
    fp = ((evC.predicted == c) & (evC.true != c)).sum()
    fn = ((evC.predicted != c) & (evC.true == c)).sum()
    rows.append((c, tp, fp, fn, round(2 * tp / (2 * tp + fp + fn), 2) if (2 * tp + fp + fn) else 0.0))
f1C = pd.DataFrame(rows, columns=["category", "tp", "fp", "fn", "f1"]).set_index("category")
print("Prompt C per-category F1"); print(f1C.to_string())

n_err = int(evC.is_error.sum())
for T in [0.40, 0.50, 0.60]:
    low = f1C.index[f1C.f1 < T].tolist()
    flag = evC.predicted.isin(low)
    caught = int((flag & evC.is_error).sum())
    print(f"\nT = {T:.2f}  flagged categories: {low}")
    print(f"  Flags:   {flag.sum()}/30 complaints ({flag.mean():.0%})")
    print(f"  Catches: {caught}/{n_err} errors ({caught / n_err:.0%}); precision of flag {caught / flag.sum():.0%}")
    print(f"  Misses:  {n_err - caught} errors")

Prompt C per-category F1
                      tp  fp  fn    f1
category                              
account_management     1   0   3  0.40
billing_dispute        1   2   3  0.29
collections_hardship   3   2   0  0.75
customer_service       0   1   3  0.00
fees_interest          3   3   1  0.60
fraud_identity         2   5   2  0.36
other                  3   1   1  0.75
rewards_marketing      2   1   2  0.57

T = 0.40  flagged categories: ['billing_dispute', 'customer_service', 'fraud_identity']
  Flags:   11/30 complaints (37%)
  Catches: 8/15 errors (53%); precision of flag 73%
  Misses:  7 errors

T = 0.50  flagged categories: ['account_management', 'billing_dispute', 'customer_service', 'fraud_identity']
  Flags:   12/30 complaints (40%)
  Catches: 8/15 errors (53%); precision of flag 67%
  Misses:  7 errors

T = 0.60  flagged categories: ['account_management', 'billing_dispute', 'customer_service', 'fraud_identity', 'rewards_marketing']
  Flags:   15/30 complaints (50%)
  Catch

## Rule 2 back-test: narrative triggers

A complaint goes to a person if its narrative is under 10 words, or if it contains instruction-like text. We check the word count on all 18,276 complaints and on the held-out set. We check the instruction screen on the 31 held-out narratives, including the adversarial one.

In [3]:
raw = pd.read_excel(RAW, usecols=["Consumer complaint narrative"])
wc_full = raw["Consumer complaint narrative"].fillna("").str.split().str.len()
print(f"(a) Full dataset: {int((wc_full < 10).sum())}/{len(wc_full)} complaints under 10 words ({(wc_full < 10).mean():.2%})")

for p in "ABC":
    ev = eval_frame(p)
    flag = ev.word_count < 10
    print(f"    Held-out, Prompt {p}: flags {int(flag.sum())}/30; catches {int((flag & ev.is_error).sum())}/{int(ev.is_error.sum())} errors; shortest narrative = {int(ev.word_count.min())} words")

pat = r"(?i)\b(?:ignore|disregard|classify (?:this|it|as)|system:|instruction)"
hit = cases.narrative.str.contains(pat, regex=True)
print(f"\n(b) Injection screen on 31 held-out narratives: flags {int(hit.sum())}/31 ({hit.mean():.0%})")
print("    Flagged complaint IDs:", cases.complaint_id[hit].tolist())
print("    Adversarial complaint caught:", bool(hit[cases.case_type == 'adversarial'].all()))
print("    Flagged typical complaints (not verified as false positives):", cases.complaint_id[hit & (cases.case_type == 'typical')].tolist())

(a) Full dataset: 73/18276 complaints under 10 words (0.40%)
    Held-out, Prompt A: flags 0/30; catches 0/15 errors; shortest narrative = 13 words
    Held-out, Prompt B: flags 0/30; catches 0/15 errors; shortest narrative = 13 words
    Held-out, Prompt C: flags 0/30; catches 0/15 errors; shortest narrative = 13 words

(b) Injection screen on 31 held-out narratives: flags 3/31 (10%)
    Flagged complaint IDs: ['9999001', '1675329', '2154642']
    Adversarial complaint caught: True
    Flagged typical complaints (not verified as false positives): ['1675329', '2154642']


## Rule 3 back-test: brief release

We block a brief if (a) any category count in it doesn't match the model's own classifications, with zero tolerance, or (b) its judge total is below 15/20. The gate value is `RUBRIC_GATE` in the code. We also check what the human means would have done under the same gate.

In [4]:
MAP = {"fraud and identity": "fraud_identity", "fees and interest": "fees_interest",
       "account management and credit reporting": "account_management", "billing disputes": "billing_dispute",
       "collections and hardship": "collections_hardship", "rewards and marketing": "rewards_marketing",
       "customer service": "customer_service", "other": "other"}

def claimed_counts(prompt):
    """Counts the brief states, from its table (Prompts B, C) or bullet list (Prompt A)."""
    out = {}
    for line in gen[prompt]["brief"].splitlines():
        m = re.match(r"\|\s*([a-z_]+)\s*\|\s*(\d+)\s*\|\s*([^|]*)\|", line)
        if m:
            out[m.group(1)] = (int(m.group(2)), set(re.findall(r"\b\d{6,8}\b", m.group(3))))
        m2 = re.match(r"-\s*([A-Za-z ]+):\s*(\d+)", line)
        if m2 and m2.group(1).strip().lower() in MAP:
            out[MAP[m2.group(1).strip().lower()]] = (int(m2.group(2)), None)
    return out

brief_flags = {}
for p in "ABC":
    preds = pd.DataFrame(gen[p]["classifications"])
    preds["id"] = preds["id"].astype(str)
    actual = preds.category.value_counts().to_dict()
    cl = claimed_counts(p)
    stated_total = sum(v[0] for v in cl.values())
    bad = [(c, cl[c][0], actual.get(c, 0)) for c in cl if cl[c][0] != actual.get(c, 0)]
    bad_ids = [c for c in cl if cl[c][1] is not None and cl[c][1] != set(preds.id[preds.category == c])]
    brief_flags[p] = {"count_mismatches": bad, "id_mismatches": bad_ids, "stated_total": stated_total}
    print(f"Prompt {p}: stated total {stated_total} (complaints: {len(preds)}); count mismatches {bad}; ID mismatches {bad_ids}")
    print("  -> rule (a) BLOCKS release" if (bad or bad_ids or stated_total != len(preds)) else "  -> rule (a) passes")

bk = pd.read_csv(H / "brief_key.csv")
bmap = dict(zip(bk.brief, bk.output))
hs = pd.concat([pd.read_csv(H / "human_scores_hunter.csv"), pd.read_csv(H / "human_scores_alettah.csv")])
hs["output"] = hs.brief.map(bmap)
human_tot = hs.groupby(["output", "rater"]).score.sum().groupby("output").mean()
judge_tot = {p: int(sum(judge[f"{p}_eval"].values())) for p in "ABC"}
# Rubric gate threshold, applied to every judge run (A/B/C _run1-3 in judge_results.json)
RUBRIC_GATE = 15
judge_runs = {p: [int(sum(judge[f"{p}_run{i}"].values())) for i in (1, 2, 3)] for p in "ABC"}
print("Judge totals per run:", judge_runs)

print()
for p in "ABC":
    print(f"Prompt {p}: judge total {judge_tot[p]}; human mean {human_tot[p]:.1f}; "
          f"rule (b) judge < {RUBRIC_GATE} on any run -> {'BLOCKS' if min(judge_runs[p]) < RUBRIC_GATE else 'passes'}; "
          f"human mean < {RUBRIC_GATE} -> {'should block' if human_tot[p] < RUBRIC_GATE else 'should pass'}")

Prompt A: stated total 32 (complaints: 31); count mismatches [('fraud_identity', 6, 5)]; ID mismatches []
  -> rule (a) BLOCKS release
Prompt B: stated total 31 (complaints: 31); count mismatches []; ID mismatches []
  -> rule (a) passes
Prompt C: stated total 31 (complaints: 31); count mismatches []; ID mismatches []
  -> rule (a) passes
Judge totals per run: {'A': [13, 12, 13], 'B': [11, 11, 11], 'C': [17, 15, 16]}

Prompt A: judge total 13; human mean 13.0; rule (b) judge < 15 on any run -> BLOCKS; human mean < 15 -> should block
Prompt B: judge total 11; human mean 13.0; rule (b) judge < 15 on any run -> BLOCKS; human mean < 15 -> should block
Prompt C: judge total 17; human mean 19.5; rule (b) judge < 15 on any run -> passes; human mean < 15 -> should pass


## Judge-human gap check

If the judge total and the human mean for a brief differ by at least the MDD, a person reviews the brief. This is what keeps the judge from signing off alone. The MDD is computed from the score files below, not typed in.

In [5]:
import sys
from itertools import combinations
sys.path.insert(0, str(ROOT / "src"))
from agreement import mdd_from

# MDD from the two raters' totals per brief (same formula as final_project.ipynb cell 48)
tot = hs.groupby(["output", "rater"]).score.sum().unstack()
pairs = list(combinations(tot.columns, 2))
total_diffs = [abs(tot.loc[o, a] - tot.loc[o, b]) for o in tot.index for a, b in pairs]
MDD = mdd_from(total_diffs)
assert round(MDD, 2) == 2.14, MDD
print(f"MDD from score files: {MDD:.2f}")
for p in "ABC":
    gap = judge_tot[p] - human_tot[p]
    print(f"Prompt {p}: judge - human = {gap:+.1f}  -> {'flag for human review' if abs(gap) >= MDD else 'within MDD'}")

MDD from score files: 2.14
Prompt A: judge - human = +0.0  -> within MDD
Prompt B: judge - human = -2.0  -> within MDD
Prompt C: judge - human = -2.5  -> flag for human review


## Kappa gate

We block autonomous routing while blind kappa on the category scheme is below 0.60. This is a gate on the whole process, not a per-complaint flag, so the back-test just reports its status. The kappa below is computed from the two labeling sheets, with the same helper as `final_project.ipynb` (cell 23). It covers 30 complaints, Hunter vs Alettah.

In [6]:
# Blind classification kappa from the two labeling sheets (same helper as final_project.ipynb cell 23)
from agreement import classification_kappa
sheets = {r: pd.read_excel(H / f"labeling_sheet_{r}.xlsx", sheet_name="labels") for r in ["hunter", "alettah"]}
assert sheets["hunter"]["label_id"].tolist() == sheets["alettah"]["label_id"].tolist()
cats = sorted(key["category"].unique())
KAPPA_BLIND = classification_kappa(
    sheets["hunter"]["category"].astype("string").str.strip(),
    sheets["alettah"]["category"].astype("string").str.strip(),
    cats,
)
print(f"Blind kappa {KAPPA_BLIND:.3f} < 0.60 -> autonomous routing BLOCKED; flags 100% of volume")

Blind kappa 0.272 < 0.60 -> autonomous routing BLOCKED; flags 100% of volume


## Data rules

Identifier scan: zero SSN-like or card-number-like strings are allowed in model inputs.

Duplicate IDs: zero duplicate complaint IDs are allowed in a batch.

In [7]:
ssn_card = r"\d{3}-\d{2}-\d{4}|(?:\d[ -]?){13,16}"
d1_hits = int(cases.narrative.str.contains(ssn_card, regex=True).sum())
print(f"Identifier scan on 31 held-out narratives: {d1_hits} matches (rule allows 0)")

raw_ids = pd.read_excel(RAW, usecols=["Complaint ID"])["Complaint ID"]
d2_dups = int(raw_ids.duplicated().sum())
print(f"Duplicate IDs on 18,276 complaints: {d2_dups} duplicate Complaint IDs (rule allows 0)")

Identifier scan on 31 held-out narratives: 0 matches (rule allows 0)


Duplicate IDs on 18,276 complaints: 0 duplicate Complaint IDs (rule allows 0)


## Summary of back-test results

These are the numbers we'll use in the memo. Every row is in-sample.

In [8]:
summary = pd.DataFrame([
    ["Low-F1 category trigger, T=0.50", "40% of held-out predictions (12/30)", "8 of 15 errors (53%)", "Kept; in-sample"],
    ["Narrative under 10 words", "0.40% of 18,276; 0 of 30 held-out", "0 of 15 errors", "Kept as guardrail; random audit added"],
    ["Injection screen", "3 of 31 (10%)", "1 of 1 adversarial; 2 unverified flags", "Kept; false positives go to review"],
    ["Count reconciliation", "1 of 3 briefs blocked (A)", "1 of 2 human-noted count errors (B's error is a Rewards line listing four IDs under '(3)')", "Prose counts outside this check; recommended for further analysis"],
    ["Judge total under 15 (any run)", "2 of 3 briefs (A, B)", "2 of 2 briefs with human mean under 15", "Threshold moved from 17 to 15 after the run check"],
    ["Judge-human gap >= 2.14", "1 of 3 briefs (C)", "1 of 1 judge miss beyond MDD", "Kept"],
], columns=["Rule", "Flag rate", "Errors caught", "Decision"])
print(summary.to_string(index=False))

                           Rule                           Flag rate                                                                              Errors caught                                                          Decision
Low-F1 category trigger, T=0.50 40% of held-out predictions (12/30)                                                                       8 of 15 errors (53%)                                                   Kept; in-sample
       Narrative under 10 words   0.40% of 18,276; 0 of 30 held-out                                                                             0 of 15 errors                             Kept as guardrail; random audit added
               Injection screen                       3 of 31 (10%)                                                     1 of 1 adversarial; 2 unverified flags                                Kept; false positives go to review
           Count reconciliation           1 of 3 briefs blocked (A) 1 of 2 human-noted count errors 